# Visualization 8: Consumption vs Contamination Analysis
## Testing the Hypothesis: Do Popular Foods Have Higher Contamination?

**Purpose:** Integrate consumption data (2017-2018) with contamination data (FY2025) to test correlation

**Data Sources:**
- Consumption: `processed_consumption_data.csv` (from Notebook 7)
- Contamination: `usda_fsis_data_product_establishment_specific_laboratory_sampling_rte_product_fy2025.json`

**Key Question:** Is there a correlation between food popularity and contamination rates?

**⚠️ DISCLAIMER:** 7-year gap between datasets (2017-18 vs 2024-25). Categories don't perfectly align. Interpret as rough comparison.

---
## Section 1: Setup and Imports

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import json
from scipy.stats import pearsonr, spearmanr
from datetime import datetime

# Set visualization style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 11

print("Libraries imported successfully!")
print(f"Analysis date: {datetime.now().strftime('%Y-%m-%d')}")

---
## Section 2: Load Both Datasets

In [ ]:
# Load consumption data (from previous notebook)
consumption_df = pd.read_csv('processed_consumption_data.csv')
print(f"✓ Consumption data loaded: {len(consumption_df):,} records")

# Load consumption summary stats
with open('consumption_summary_stats.json', 'r') as f:
    consumption_stats = json.load(f)

print("\nConsumption Summary:")
print(f"  Cured meat (2017): {consumption_stats['cured_meat']['recent_lbs_per_year']:.1f} lbs/year")
print(f"  Peak year: {consumption_stats['cured_meat']['peak_year']}")
print(f"  Decline: {abs(consumption_stats['cured_meat']['decline_percent']):.1f}%")

In [ ]:
# Load contamination data
json_path = 'usda_fsis_data_product_establishment_specific_laboratory_sampling_rte_product_fy2025.json'

with open(json_path, 'r') as f:
    fsis_data = json.load(f)[0]

contamination_df = pd.DataFrame(fsis_data['data']['primary_table_data'])
print(f"\n✓ Contamination data loaded: {len(contamination_df):,} samples")
print(f"  Columns: {len(contamination_df.columns)}")

---
## Section 3: Categorize Contamination Data by Protein Type

In [ ]:
# Filter for samples with Listeria test results
contamination_clean = contamination_df[
    contamination_df['lm_listeria_analysis'].notna()
].copy()

print(f"Samples with Listeria results: {len(contamination_clean):,}")

In [ ]:
def categorize_protein_type(source_name):
    """
    Categorize sample by protein type
    
    Mapping:
    - Chicken/Turkey -> Poultry
    - Pork + Sausage -> Cured Meat (Sausage)
    - Pork (other) -> Pork
    - Beef -> Beef
    - Contact surface -> Environmental
    """
    if pd.isna(source_name):
        return 'Unknown'
    
    source = str(source_name).lower()
    
    if 'chicken' in source or 'turkey' in source:
        return 'Poultry'
    elif 'pork' in source and 'sausage' in source:
        return 'Cured Meat (Sausage)'
    elif 'pork' in source:
        return 'Pork'
    elif 'beef' in source:
        return 'Beef'
    elif 'sausage' in source:
        return 'Cured Meat (Sausage)'
    elif 'contact' in source:
        return 'Environmental'
    else:
        return 'Other'

contamination_clean['protein_category'] = contamination_clean['sample_source_name'].apply(categorize_protein_type)
contamination_clean['is_positive'] = contamination_clean['lm_listeria_analysis'] == 'Positive'

print("\nProtein categories identified:")
print(contamination_clean['protein_category'].value_counts())

---
## Section 4: Calculate Contamination Rates by Category

In [ ]:
# Exclude environmental samples for product analysis
contamination_products = contamination_clean[
    contamination_clean['protein_category'] != 'Environmental'
].copy()

# Calculate rates by category
contamination_rates = contamination_products.groupby('protein_category').agg({
    'is_positive': ['sum', 'count', 'mean']
}).reset_index()

contamination_rates.columns = ['protein_category', 'positive_count', 'total_samples', 'contamination_rate']
contamination_rates['contamination_rate_pct'] = contamination_rates['contamination_rate'] * 100
contamination_rates = contamination_rates.sort_values('contamination_rate_pct', ascending=False)

print("\nContamination Rates by Protein Category:")
print("="*80)
for _, row in contamination_rates.iterrows():
    cat = row['protein_category']
    pos = int(row['positive_count'])
    total = int(row['total_samples'])
    rate = row['contamination_rate_pct']
    print(f"{cat:30s} {rate:6.2f}% ({pos:3d}/{total:4d} samples)")
print("="*80)

---
## Section 5: Create Matched Dataset (Consumption + Contamination)

In [ ]:
# Helper function to safely get contamination rate
def get_contamination_rate(category):
    match = contamination_rates[contamination_rates['protein_category'] == category]
    if len(match) > 0:
        return {
            'rate_pct': float(match['contamination_rate_pct'].values[0]),
            'samples': int(match['total_samples'].values[0]),
            'positives': int(match['positive_count'].values[0])
        }
    return {'rate_pct': 0.0, 'samples': 0, 'positives': 0}

# Get consumption values for 2017
poultry_oz = consumption_stats['categories_2017']['poultry_oz_per_day']
meats_oz = consumption_stats['categories_2017']['meats_oz_per_day']
cured_oz = consumption_stats['categories_2017']['cured_meat_oz_per_day']

# Create matched dataset
matched_data = []

# Poultry (Good match)
poultry_contam = get_contamination_rate('Poultry')
matched_data.append({
    'category': 'Poultry',
    'consumption_oz_day': poultry_oz,
    'consumption_lbs_year': poultry_oz * 365 / 16,
    'contamination_rate_pct': poultry_contam['rate_pct'],
    'total_samples': poultry_contam['samples'],
    'positive_samples': poultry_contam['positives'],
    'match_quality': 'Good',
    'notes': 'Direct category match'
})

# Cured Meat/Sausage (Moderate match)
sausage_contam = get_contamination_rate('Cured Meat (Sausage)')
matched_data.append({
    'category': 'Cured Meat (RTE Sausage)',
    'consumption_oz_day': cured_oz,
    'consumption_lbs_year': cured_oz * 365 / 16,
    'contamination_rate_pct': sausage_contam['rate_pct'],
    'total_samples': sausage_contam['samples'],
    'positive_samples': sausage_contam['positives'],
    'match_quality': 'Moderate',
    'notes': 'Sausage is subset of all cured meat'
})

# Pork (Poor match - consumption includes beef/lamb)
pork_contam = get_contamination_rate('Pork')
matched_data.append({
    'category': 'Pork (Non-Sausage)',
    'consumption_oz_day': meats_oz,
    'consumption_lbs_year': meats_oz * 365 / 16,
    'contamination_rate_pct': pork_contam['rate_pct'],
    'total_samples': pork_contam['samples'],
    'positive_samples': pork_contam['positives'],
    'match_quality': 'Poor',
    'notes': 'Consumption includes beef/lamb too'
})

# Beef (Poor match)
beef_contam = get_contamination_rate('Beef')
matched_data.append({
    'category': 'Beef',
    'consumption_oz_day': meats_oz,
    'consumption_lbs_year': meats_oz * 365 / 16,
    'contamination_rate_pct': beef_contam['rate_pct'],
    'total_samples': beef_contam['samples'],
    'positive_samples': beef_contam['positives'],
    'match_quality': 'Poor',
    'notes': 'Consumption includes pork/lamb too'
})

matched_df = pd.DataFrame(matched_data)

print("\n✓ Matched Dataset Created")
print("\nMatched Categories:")
print(matched_df[['category', 'consumption_lbs_year', 'contamination_rate_pct', 'match_quality']])

---
## Section 6: Visualization 1 - Scatter Plot (Consumption vs Contamination)

In [ ]:
# Create scatter plot
fig, ax = plt.subplots(figsize=(14, 9))

# Color mapping
color_map = {
    'Good': '#2ca02c',
    'Moderate': '#ff7f0e',
    'Poor': '#d62728'
}

# Plot each point
for match_quality in matched_df['match_quality'].unique():
    subset = matched_df[matched_df['match_quality'] == match_quality]
    ax.scatter(
        subset['consumption_lbs_year'],
        subset['contamination_rate_pct'],
        s=subset['total_samples'] * 2,  # Bubble size = sample count
        alpha=0.6,
        c=color_map.get(match_quality, '#7f7f7f'),
        edgecolors='black',
        linewidth=2.5,
        label=f'{match_quality} Match'
    )

# Add labels for each point
for _, row in matched_df.iterrows():
    ax.annotate(
        row['category'],
        xy=(row['consumption_lbs_year'], row['contamination_rate_pct']),
        xytext=(8, 8),
        textcoords='offset points',
        fontsize=11,
        fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.8)
    )

# Labels and title
ax.set_xlabel('Consumption (pounds per person per year)', fontsize=13, fontweight='bold')
ax.set_ylabel('Contamination Rate (%)', fontsize=13, fontweight='bold')
ax.set_title(
    'Consumption vs Contamination: Do Popular Foods Have Higher Contamination?\n' +
    'Bubble Size = Sample Count | Color = Data Match Quality',
    fontsize=15, fontweight='bold', pad=20
)

ax.legend(title='Match Quality', loc='upper right', fontsize=11, title_fontsize=12)
ax.grid(True, alpha=0.3)

# Add disclaimer
fig.text(
    0.5, 0.02,
    '⚠️ DISCLAIMER: Consumption data (2017-18) vs Contamination data (2024-25). ' +
    '7-year gap. Categories do not perfectly align. Interpret as rough comparison.',
    ha='center', fontsize=10, style='italic', color='red', weight='bold'
)

plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.savefig('consumption_vs_contamination_scatter.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Chart saved: consumption_vs_contamination_scatter.png")

---
## Section 7: Statistical Correlation Analysis

In [ ]:
# Filter for good matches only for statistical test
good_matches = matched_df[matched_df['match_quality'] == 'Good']

print("="*80)
print("CORRELATION ANALYSIS")
print("="*80)

if len(good_matches) >= 2:
    # Calculate Pearson correlation
    r_pearson, p_pearson = pearsonr(
        good_matches['consumption_lbs_year'],
        good_matches['contamination_rate_pct']
    )
    
    # Calculate Spearman correlation (rank-based)
    r_spearman, p_spearman = spearmanr(
        good_matches['consumption_lbs_year'],
        good_matches['contamination_rate_pct']
    )
    
    print(f"\nUsing {len(good_matches)} 'Good' quality matches:")
    for _, row in good_matches.iterrows():
        print(f"  - {row['category']}: {row['consumption_lbs_year']:.1f} lbs/year, "
              f"{row['contamination_rate_pct']:.2f}% contamination")
    
    print(f"\nPearson Correlation:  r = {r_pearson:.3f}, p-value = {p_pearson:.3f}")
    print(f"Spearman Correlation: rho = {r_spearman:.3f}, p-value = {p_spearman:.3f}")
    
    # Interpretation
    print("\n" + "="*80)
    print("INTERPRETATION:")
    print("="*80)
    
    if p_pearson < 0.05:
        if r_pearson > 0.7:
            conclusion = "STRONG POSITIVE CORRELATION: Popular foods have significantly higher contamination"
        elif r_pearson > 0.4:
            conclusion = "MODERATE POSITIVE CORRELATION: Some evidence that popular foods have higher contamination"
        elif r_pearson > 0:
            conclusion = "WEAK POSITIVE CORRELATION: Minimal evidence of relationship"
        else:
            conclusion = "NEGATIVE CORRELATION: Popular foods have LOWER contamination"
    else:
        conclusion = "NO SIGNIFICANT CORRELATION: Consumption does not predict contamination rate"
    
    print(f"\n{conclusion}")
    print("\nNote: Limited to 'Good' quality matches only. More data needed for robust conclusion.")
    
    # Save results
    correlation_results = {
        'analysis_date': datetime.now().strftime('%Y-%m-%d'),
        'n_good_matches': len(good_matches),
        'pearson_r': float(r_pearson),
        'pearson_p': float(p_pearson),
        'spearman_rho': float(r_spearman),
        'spearman_p': float(p_spearman),
        'conclusion': conclusion
    }
    
else:
    print("\n⚠️  Not enough good-quality matches for statistical correlation test")
    print(f"   Need at least 2, have {len(good_matches)}")
    correlation_results = None

print("="*80)

---
## Section 8: Visualization 2 - Comparison Table

In [ ]:
# Create comparison table visualization
fig, ax = plt.subplots(figsize=(16, 8))
ax.axis('tight')
ax.axis('off')

# Prepare table data
table_data = []
table_data.append([
    'Category',
    'Consumption\n(lbs/year)',
    'Contamination\nRate (%)',
    'Samples\n(Pos/Total)',
    'Match\nQuality',
    'Notes'
])

for _, row in matched_df.iterrows():
    table_data.append([
        row['category'],
        f"{row['consumption_lbs_year']:.1f}",
        f"{row['contamination_rate_pct']:.2f}%",
        f"{int(row['positive_samples'])}/{int(row['total_samples'])}",
        row['match_quality'],
        row['notes']
    ])

table = ax.table(
    cellText=table_data,
    cellLoc='left',
    loc='center',
    colWidths=[0.20, 0.12, 0.12, 0.12, 0.10, 0.34]
)

table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 2.5)

# Style header row
for i in range(6):
    table[(0, i)].set_facecolor('#4CAF50')
    table[(0, i)].set_text_props(weight='bold', color='white', fontsize=11)

# Color rows by match quality
colors = {'Good': '#c8e6c9', 'Moderate': '#fff9c4', 'Poor': '#ffccbc'}
for i in range(1, len(table_data)):
    match_quality = table_data[i][4]
    color = colors.get(match_quality, 'white')
    for j in range(6):
        table[(i, j)].set_facecolor(color)

plt.title(
    'Consumption vs Contamination: Category Comparison\n' +
    'Green = Good Match | Yellow = Moderate Match | Red = Poor Match',
    fontsize=15, fontweight='bold', pad=20
)

plt.savefig('consumption_vs_contamination_table.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Chart saved: consumption_vs_contamination_table.png")

---
## Section 9: Export Results

In [ ]:
# Export matched dataset
output_csv = 'consumption_vs_contamination_matched.csv'
matched_df.to_csv(output_csv, index=False)
print(f"✓ Matched dataset exported: {output_csv}")

# Export correlation results
if correlation_results:
    output_json = 'correlation_results.json'
    with open(output_json, 'w') as f:
        json.dump(correlation_results, f, indent=2)
    print(f"✓ Correlation results exported: {output_json}")

---
## Section 10: Summary and Key Findings

In [ ]:
print("="*80)
print("ANALYSIS COMPLETE: Consumption vs Contamination")
print("="*80)

print("\n📊 VISUALIZATIONS CREATED:")
print("   1. consumption_vs_contamination_scatter.png - Scatter plot with match quality")
print("   2. consumption_vs_contamination_table.png - Detailed comparison table")

print("\n💾 DATA EXPORTED:")
print("   1. consumption_vs_contamination_matched.csv - Matched dataset")
if correlation_results:
    print("   2. correlation_results.json - Statistical results")

print("\n🔍 KEY FINDINGS:")
print("\n1. CONSUMPTION PATTERNS (2017-2018):")
for _, row in matched_df.sort_values('consumption_lbs_year', ascending=False).iterrows():
    print(f"   • {row['category']:30s} {row['consumption_lbs_year']:6.1f} lbs/year")

print("\n2. CONTAMINATION PATTERNS (FY2025):")
for _, row in matched_df.sort_values('contamination_rate_pct', ascending=False).iterrows():
    print(f"   • {row['category']:30s} {row['contamination_rate_pct']:5.2f}% "
          f"({int(row['positive_samples'])}/{int(row['total_samples'])} samples)")

print("\n3. CORRELATION:")
if correlation_results:
    print(f"   • {correlation_results['conclusion']}")
    print(f"   • Pearson r = {correlation_results['pearson_r']:.3f}, "
          f"p = {correlation_results['pearson_p']:.3f}")
else:
    print("   • Insufficient data for statistical correlation test")

print("\n4. LIMITATIONS:")
print("   ⚠️  7-year gap between datasets (2017-18 vs 2024-25)")
print("   ⚠️  Category mismatch: Cured meat ⊃ RTE Sausage")
print("   ⚠️  Consumption includes raw + RTE for some categories")
print("   ⚠️  Limited sample size for robust statistical conclusions")

print("\n➡️  NEXT STEP:")
print("   Run visualization_9_state_map_choropleth.ipynb")
print("   to create interactive state contamination map")

print("="*80)